In [10]:
import numpy as np
from numpy.polynomial.legendre import leggauss
from scipy import integrate
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.Utils import (
    compute_jacobian_XoE,
    CanonicalUnits, 
    GravitationalParameters, 
    trasformation_X_to_E,
    P_E, 
    P_X_vectorized,
    )

In [11]:
cu = CanonicalUnits()
deg = cu.deg
AU_m = cu.AU_m #m
M_sun = cu.M_sun
G = cu.G # m^3 / (kg s^2)
year = cu.year #s
mu = cu.mu
grav_params = GravitationalParameters(mu=mu)

In [12]:
def P_X(x: float, y: float, z: float, vx: float, vy: float, vz: float) -> float:
    a, e, i, Omega, w, M = trasformation_X_to_E(x, y, z, vx, vy, vz, mu)
    J = compute_jacobian_XoE(a,e,i,Omega,w,M,mu)
    det = np.linalg.det(J)
    #print(det)
    inv_det = 1.0/det
    P = P_E() * abs(inv_det)
    return mu

In [15]:
# Define center and widths of the phase-space hypercube
c_x = 1
c_y = 0
c_z = 0
v_x = 0
v_y = (mu/1)**0.5
v_z = 0

dxyz = 0.2
dvxyz = 5000 * (1/AU_m) * year 

center = (c_x, c_y, c_z, v_x, v_y, v_z)
widths = (dxyz, dxyz, dxyz, dvxyz, dvxyz, dvxyz)


In [17]:
# Integration bounds for each variable (modify to your domain)
bounds = {
    'x1': (center[0] - 0.5*widths[0], center[0] + 0.5*widths[0]),
    'x2': (center[1] - 0.5*widths[1], center[1] + 0.5*widths[1]),
    'x3': (center[2] - 0.5*widths[2], center[2] + 0.5*widths[2]),
    'x4': (center[3] - 0.5*widths[3], center[3] + 0.5*widths[3]),
    'x5': (center[4] - 0.5*widths[4], center[4] + 0.5*widths[4]),
    'z' : (center[5] - 0.5*widths[5], center[5] + 0.5*widths[5])   # problematic dimension
}

# Number of Gauss-Legendre nodes per smooth dimension (tune)
N_gl = 8 

# Build GL nodes and weights mapped to [a,b]
def gl_nodes_weights(a, b, N):
    nodes, weights = leggauss(N)
    # map from [-1,1] to [a,b]
    mapped_nodes = 0.5*(nodes + 1)*(b - a) + a
    mapped_weights = 0.5*(b - a) * weights
    return mapped_nodes, mapped_weights

x1_nodes, x1_w = gl_nodes_weights(*bounds['x1'], N_gl)
x2_nodes, x2_w = gl_nodes_weights(*bounds['x2'], N_gl)
x3_nodes, x3_w = gl_nodes_weights(*bounds['x3'], N_gl)
x4_nodes, x4_w = gl_nodes_weights(*bounds['x4'], N_gl)
x5_nodes, x5_w = gl_nodes_weights(*bounds['x5'], N_gl)

# Create outer grid (cartesian product)
# If N_gl^5 is too large, reduce N_gl or use sparse grids.
grid = np.array(np.meshgrid(x1_nodes, x2_nodes, x3_nodes, x4_nodes, x5_nodes, indexing='ij'))
# grid.shape = (5, N, N, N, N, N). We'll reshape to list of points.
num_outer = grid.reshape(5, -1).shape[1]
outer_points = grid.reshape(5, -1).T  # shape: (num_outer, 5)

# Compute outer weights as tensor product
w_grid = np.array(np.meshgrid(x1_w, x2_w, x3_w, x4_w, x5_w, indexing='ij'))
outer_weights = w_grid.reshape(5, -1).prod(axis=0)  # length num_outer

z_a, z_b = bounds['z']

# define inner integrand as function of z and outer_point tuple
def inner_integral_for_outer(point5):
    x1, x2, x3, x4, x5 = point5
    # scalar function of z (for scipy.integrate.quad)
    def fz(z):
        return P_X(x1, x2, x3, x4, x5, z)
    # integrate with adaptive Gauss-Kronrod (quad)
    val, err = integrate.quad(fz, z_a, z_b, epsabs=1e-8, epsrel=1e-8, limit=200)
    return val

# Parallel evaluation over outer points
def compute_total_integral(num_workers=8):
    total = 0.0
    # Use ThreadPoolExecutor because scipy.quad releases GIL in C; if your integrator is Python-level heavy,
    # consider ProcessPoolExecutor instead.
    with ThreadPoolExecutor(max_workers=num_workers) as ex:
        futures = {ex.submit(inner_integral_for_outer, tuple(outer_points[i])): i for i in range(num_outer)}
        for fut in as_completed(futures):
            i = futures[fut]
            inner_val = fut.result()
            total += outer_weights[i] * inner_val
    return total


In [19]:
N_theoretical = compute_total_integral(num_workers=8)

In [20]:
N = int(1e7)
print(f"Theoretical (integral) number of objects in volume: {N_theoretical * N}")

Theoretical (integral) number of objects in volume: 3706744.188219675


In [21]:
import numpy as np
from numpy.polynomial.legendre import leggauss
from scipy.integrate import quad

def hybrid_integral_P_X(center, widths, n_points=6, mu=1):
    """
    Hybrid 6D integral of P_X:
    - 5D Gauss-Legendre quadrature for smooth coords (x, y, z, vx, vy)
    - 1D adaptive integration (scipy.quad) for problematic coordinate (vz)

    Parameters
    ----------
    center : (x, y, z, vx, vy, vz)
        Center of the hypercube in phase space.
    widths : (dx, dy, dz, dvx, dvy, dvz)
        Side lengths in each dimension.
    n_points : int
        Number of Gauss-Legendre nodes for each smooth dimension.
    mu : float
        Parameter passed to P_X_vectorized.

    Returns
    -------
    float
        Approximate volume integral of P_X.
    """

    # Unpack inputs
    x0, y0, z0, vx0, vy0, vz0 = center
    dx, dy, dz, dvx, dvy, dvz = widths

    # Get Gauss-Legendre points & weights for [-1,1]
    pts, wts = leggauss(n_points)

    # Map to actual intervals for the *smooth* dims
    def map_pts(c, w): return c + 0.5 * w * pts
    def map_wts(w): return 0.5 * w * wts

    x_pts, y_pts, z_pts, vx_pts, vy_pts = map_pts(x0, dx), map_pts(y0, dy), map_pts(z0, dz), map_pts(vx0, dvx), map_pts(vy0, dvy)
    wx, wy, wz, wvx, wvy = map_wts(dx), map_wts(dy), map_wts(dz), map_wts(dvx), map_wts(dvy)

    # Create meshgrid for 5D outer quadrature
    X, Y, Z, VX, VY = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY = np.meshgrid(wx, wy, wz, wvx, wvy, indexing='ij')

    # Flatten
    Xf, Yf, Zf, VXf, VYf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel()
    WF = (WX * WY * WZ * WVX * WVY).ravel()

    # Bounds for vz
    vz_min, vz_max = vz0 - dvz / 2, vz0 + dvz / 2

    total = 0.0
    for i in range(len(WF)):
        x, y, z, vx, vy = Xf[i], Yf[i], Zf[i], VXf[i], VYf[i]

        # Define 1D integrand for the problematic dimension vz
        def integrand_vz(vz):
            return P_X_vectorized(x, y, z, vx, vy, vz, mu)

        # Adaptive integration over vz
        inner_val, err = quad(integrand_vz, vz_min, vz_max, epsabs=1e-8, epsrel=1e-8, limit=200)
        total += WF[i] * inner_val

    return total


In [22]:
c_x = 1
c_y = 0
c_z = 0
v_x = 0
v_y = (mu/1)**0.5
v_z = 0

dxyz = 0.2
dvxyz = 5000 * (1/AU_m) * year 

center = (c_x, c_y, c_z, v_x, v_y, v_z)
widths = (dxyz, dxyz, dxyz, dvxyz, dvxyz, dvxyz)

# Theoretical: compute expected number of objects in the volume by integrating the distribution
N_theoretical = hybrid_integral_P_X(center, widths, n_points=8, mu=mu)
print(f"Theoretical (integral) number of objects in volume: {N_theoretical * N}")

Theoretical (integral) number of objects in volume: 106.0664242281959


In [25]:
import numpy as np
from numpy.polynomial.legendre import leggauss
from scipy.integrate import quad

def hybrid_integral_P_X_ultra(center, widths, n_points=6, mu=1):
    """
    Hybrid 6D integral of P_X:
      - 5D Gauss-Legendre quadrature for smooth coords (x, y, z, vx, vy)
      - 1D adaptive integration (scipy.quad) for problematic coordinate vz
        tuned for extremely sharp peaks around vz≈0
    """

    x0, y0, z0, vx0, vy0, vz0 = center
    dx, dy, dz, dvx, dvy, dvz = widths

    pts, wts = leggauss(n_points)
    def map_pts(c, w): return c + 0.5 * w * pts
    def map_wts(w): return 0.5 * w * wts

    x_pts, y_pts, z_pts, vx_pts, vy_pts = (
        map_pts(x0, dx), map_pts(y0, dy), map_pts(z0, dz),
        map_pts(vx0, dvx), map_pts(vy0, dvy)
    )
    wx, wy, wz, wvx, wvy = (
        map_wts(dx), map_wts(dy), map_wts(dz),
        map_wts(dvx), map_wts(dvy)
    )

    X, Y, Z, VX, VY = np.meshgrid(
        x_pts, y_pts, z_pts, vx_pts, vy_pts, indexing='ij'
    )
    WX, WY, WZ, WVX, WVY = np.meshgrid(
        wx, wy, wz, wvx, wvy, indexing='ij'
    )

    Xf, Yf, Zf, VXf, VYf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel()
    WF = (WX * WY * WZ * WVX * WVY).ravel()

    vz_min, vz_max = vz0 - dvz/2, vz0 + dvz/2

    # --- tuned parameters ---
    epsabs = 1e-12
    epsrel = 1e-9
    limit  = 800
    delta  = 0.02 * dvz   # half-width of fine region around vz≈0

    total = 0.0
    for i in range(len(WF)):
        x, y, z, vx, vy = Xf[i], Yf[i], Zf[i], VXf[i], VYf[i]

        def f_vz(vz):
            return P_X_vectorized(x, y, z, vx, vy, vz, mu)

        # integrate in 3 segments around the peak
        val = 0.0
        segments = [
            (vz_min, -delta),
            (-delta,  delta),
            ( delta,  vz_max)
        ]
        for a, b in segments:
            if a < b:  # avoid inverted segments
                v, _ = quad(f_vz, a, b, epsabs=epsabs, epsrel=epsrel, limit=limit)
                val += v

        total += WF[i] * val

    return total


In [24]:
c_x = 1
c_y = 0
c_z = 0
v_x = 0
v_y = (mu/1)**0.5
v_z = 0

dxyz = 0.2
dvxyz = 5000 * (1/AU_m) * year 

center = (c_x, c_y, c_z, v_x, v_y, v_z)
widths = (dxyz, dxyz, dxyz, dvxyz, dvxyz, dvxyz)

# Theoretical: compute expected number of objects in the volume by integrating the distribution
N_theoretical =hybrid_integral_P_X_ultra(center, widths, n_points=8, mu=mu)
print(f"Theoretical (integral) number of objects in volume: {N_theoretical * N}")

Theoretical (integral) number of objects in volume: 106.0664242282497


In [ ]:
import numpy as np
from numpy.polynomial.legendre import leggauss
from scipy.integrate import quad, _quadpack
from scipy import integrate

def hybrid_integral_P_X_qagpe(center, widths, n_points=6, mu=1):
    """
    6D hybrid integration of P_X using:
      - Gauss-Legendre (n_points) for 5 smooth variables
      - QAGPE (explicit singularity points) for vz
    """

    x0, y0, z0, vx0, vy0, vz0 = center
    dx, dy, dz, dvx, dvy, dvz = widths

    # Gauss-Legendre setup
    pts, wts = leggauss(n_points)
    def map_pts(c, w): return c + 0.5 * w * pts
    def map_wts(w): return 0.5 * w * wts

    x_pts, y_pts, z_pts, vx_pts, vy_pts = (
        map_pts(x0, dx), map_pts(y0, dy), map_pts(z0, dz),
        map_pts(vx0, dvx), map_pts(vy0, dvy)
    )
    wx, wy, wz, wvx, wvy = (
        map_wts(dx), map_wts(dy), map_wts(dz),
        map_wts(dvx), map_wts(dvy)
    )

    X, Y, Z, VX, VY = np.meshgrid(
        x_pts, y_pts, z_pts, vx_pts, vy_pts, indexing='ij'
    )
    WX, WY, WZ, WVX, WVY = np.meshgrid(
        wx, wy, wz, wvx, wvy, indexing='ij'
    )

    Xf, Yf, Zf, VXf, VYf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel()
    WF = (WX * WY * WZ * WVX * WVY).ravel()

    vz_min, vz_max = vz0 - dvz/2, vz0 + dvz/2

    # Define “trouble points” where integrator should refine
    # (for example: sharp peak around 0 and 0.2)
    points = [0.0, 0.1, 0.2]

    # Integrator parameters
    epsabs = 1e-12
    epsrel = 1e-9
    limit = 900
    #key = 6  # QAGPE rule selector (Gauss-Kronrod 61)

    total = 0.0
    for i in range(len(WF)):
        x, y, z, vx, vy = Xf[i], Yf[i], Zf[i], VXf[i], VYf[i]

        def f_vz(vz):
            return P_X_vectorized(x, y, z, vx, vy, vz, mu)

        # Call the low-level QAGPE routine directly
        #inner_val, err = quad(integrand_vz, vz_min, vz_max, epsabs=1e-8, epsrel=1e-8, limit=200)
        val, err = quad(
            f_vz, vz_min, vz_max, points=points,
            epsabs=epsabs, epsrel=epsrel, limit=limit
        )
        total += WF[i] * val

    return total


In [44]:
c_x = 1
c_y = 0
c_z = 0
v_x = 0
v_y = (mu/1)**0.5
v_z = 0

dxyz = 0.2
dvxyz = 5000 * (1/AU_m) * year 

center = (c_x, c_y, c_z, v_x, v_y, v_z)
widths = (dxyz, dxyz, dxyz, dvxyz, dvxyz, dvxyz)

# Theoretical: compute expected number of objects in the volume by integrating the distribution
N_theoretical = hybrid_integral_P_X_qagpe(center, widths, n_points=8, mu=mu)
print(f"Theoretical (integral) number of objects in volume: {N_theoretical * N}")

Theoretical (integral) number of objects in volume: 106.06642422824966
